In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import argparse
import numpy as np
import os
from qonnx.core.datatype import DataType
from driver_base import FINNExampleOverlay
from pynq.pl_server.device import Device

In [ ]:
def block(self, ind=0):
    return io_shape_dict["block"][ind]

In [ ]:
import os
import numpy as np

def load_weights():
    w_filenames = []
    runtime_weight_dir = "MVAU_weights_npy/"
    if not os.path.isdir(runtime_weight_dir):
        return
    for dirpath, dirnames, filenames in os.walk(runtime_weight_dir):
        w_filenames.extend(filenames)

    w_filenames = sorted(w_filenames, key=lambda x: int(x.split('_')[1].split('.')[0]))

    weight_tensor_dconv = []
    weight_tensor_ups = []
    
    for w_filename in w_filenames:
        if w_filename.endswith(".npy"):
            index = int(w_filename.split('_')[1].split('.')[0])
            #print(index)
            weight_tensor = np.load(runtime_weight_dir + "/" + w_filename)
            if index in [6, 9]:
                weight_tensor_ups.append(weight_tensor)
                print("%s : index %d, weight_tensor_shape %s" % (w_filename, index, weight_tensor.shape))
            else:
                weight_tensor_dconv.append(weight_tensor)
                #print("%s : index %d, weight_tensor_shape %s" % (w_filename, index, weight_tensor.shape))
        else:
            continue

        
    print(f"Collected filenames: {w_filenames}")
   

load_weights()

In [ ]:
# dictionary describing the I/O of the FINN-generated accelerator
fmdim = 6
io_shape_dict = {
    # FINN DataType for input and output tensors
    "idt" : [DataType['UINT4']],
    "odt" : [DataType['INT32']],
    # shapes for input and output tensors (NHWC layout)
    "ishape_normal" : [(1, fmdim, fmdim, 2)],
    "oshape_normal" : [(1, fmdim, fmdim, 1)],
    # folded / packed shapes below depend on idt/odt and input/output
    # PE/SIMD parallelization settings -- these are calculated by the
    # FINN compiler.
    "ishape_folded" : [(1, fmdim, fmdim, 1, 2)],
    "oshape_folded" : [(1, fmdim, fmdim, 1, 1)],
    "ishape_packed" : [(1, fmdim, fmdim, 1, 1)],
    "oshape_packed" : [(1, fmdim, fmdim, 1, 4)], # the last dimension should be the numBytes(8 bits)
    "input_dma_name" : ['idma_0'],
    "weight_dma_name" : ['idma_weight_1_0', 'idma_weight_2_0'],
    "accel_name" : ['top_0'],
    "output_dma_name" : ['odma_0'],
    # "number_of_external_weights": 0,
    "num_inputs" : 1,
    "num_outputs" : 1,
    # args for accel
    # (block_r, True/False) indicates if there is the second weight load operation in that block, block_r is the block number
    "block" : [(i, i in [6, 8]) for i in range(1, 11)],
    "IFMDim_arg": [6, 6, 3, 3, 1, 1, 3, 3, 6, 6],
    "OFMDim_arg": [6, 3, 3, 1, 1, 3, 3, 6, 6, 6],
    "IFMChannel_arg": [2, 16, 16, 32, 32, 64, 32, 16, 16, 8],
    "MVAU_OFMChannel_arg": [16, 16, 32, 32, 64, 64, 16, 16, 8, 8],
    "weight_in_simd_arg": [8, 16, 16, 32, 32, 64, 32, 16, 16, 8],
    "MVAU_Tiles_arg": [9, 9, 9, 9, 18, 18, 9, 9, 9, 9],
    "UpS_Tiles_arg": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
    "OUPChannel_arg": [0, 0, 0, 0, 0, 32, 0, 16, 0, 0],
    "nf_compute": [1, 1, 1, 1, 2, 2, 1, 1, 1, 1],
    "scale_factor_arg": [0, 0, 0, 0, 0, 3, 0, 2, 0, 0],
    "Padding_arg": [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    "MaxPooling_en": [0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
    "Upsampling_en": [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
    "buf_index": [0, 0, 0, 1, 0, 1, 0, 0, 0, 0]
}

exec_mode = "throughput_test"
platform = "zynq-iodma"
batch_size = 1
fclk_mhz = 250
bitfile = "unet3L.bit"
runtime_weight_dir = "MVAU_weights_npy/"
devID = 0
device = Device.devices[devID]

# instantiate FINN accelerator driver and pass batchsize and bitfile
accel = FINNExampleOverlay(
    bitfile_name = bitfile, platform = platform,
    io_shape_dict = io_shape_dict, batch_size = batch_size, fclk_mhz = fclk_mhz,
    runtime_weight_dir = runtime_weight_dir, device=device
)


In [ ]:
accel.ip_dict

In [ ]:
from pynq import GPIO

# Define a function to read specific bit ranges from GPIO
def read_gpio_ranges():
    # Read bits 11:0
    gpio_pins_0_11 = [GPIO(GPIO.get_gpio_pin(i), 'in') for i in range(12)]
    value_0_11 = 0
    for i in range(12):
        bit_value = gpio_pins_0_11[i].read()
        value_0_11 |= (bit_value << i)

    # Read bits 23:12
    gpio_pins_12_23 = [GPIO(GPIO.get_gpio_pin(i), 'in') for i in range(12, 24)]
    value_12_23 = 0
    for i in range(12):
        bit_value = gpio_pins_12_23[i].read()
        value_12_23 |= (bit_value << i)

    # Read bits 29:24
    gpio_pins_24_29 = [GPIO(GPIO.get_gpio_pin(i), 'in') for i in range(24, 30)]
    value_24_29 = 0
    for i in range(6):
        bit_value = gpio_pins_24_29[i].read()
        value_24_29 |= (bit_value << i)

    # Read bits 35:30
    gpio_pins_30_35 = [GPIO(GPIO.get_gpio_pin(i), 'in') for i in range(30, 36)]
    value_30_35 = 0
    for i in range(6):
        bit_value = gpio_pins_30_35[i].read()
        value_30_35 |= (bit_value << i)

    # Return all the values
    return value_0_11, value_12_23, value_24_29, value_30_35

In [ ]:
import pynq
import time

rails = pynq.get_rails()
recorder = pynq.DataRecorder(rails['VCCp0V85'].power)

In [ ]:
# for the remote execution the data from the input npy file has to be loaded,
# packed and copied to the PYNQ buffer
exec_mode = "execute"
total = 0
if exec_mode == "execute":
    # load desired input .npy file(s)
    ibuf_normal = []
    for ifn in range(0, 1):
        ibuf_normal.append(arr)
    recorder.reset()
    with recorder.record(0.005):
        time.sleep(3)
        for _ in range(100):
            obuf_normal, run_once_time = accel.execute(ibuf_normal)
            total = total + run_once_time
        time.sleep(3)
#     obuf_normal = accel.execute(ibuf_normal) # this execute function should run all the blocks(a complete unet), 
    average = total/100   # meaning that the accel is actually running multiple times in the execute function
    if not isinstance(obuf_normal, list):
        obuf_normal = [obuf_normal]
#     for o, obuf in enumerate(obuf_normal):
#         np.save(outputfile[o], obuf)
elif exec_mode == "throughput_test":
    # remove old metrics file
    try:
        os.remove("nw_metrics.txt")
    except FileNotFoundError:
        pass
#     recorder.reset()
#     with recorder.record(0.005):
#         time.sleep(10)
#         for _ in range(1):
#             res = accel.throughput_test()
#         time.sleep(10)
    accel.throughput_test()
else:
    raise Exception("Exec mode has to be set to execute or throughput_test")

In [ ]:
%matplotlib inline
recorder.frame['VCCp0V85_power'].plot()

In [ ]:
max_power = recorder.frame['VCCp0V85_power'].max()
print(max_power)

In [ ]:
value_0_11, value_12_23, value_24_29, value_30_35 = read_gpio_ranges()
print(f"Value of bits 11:0: {value_0_11}")
print(f"Value of bits 23:12: {value_12_23}")
print(f"Value of bits 29:24: {value_24_29}")
print(f"Value of bits 35:30: {value_30_35}")

In [ ]:
'''
{
    'VCCpSYZYGY_VIO': 
        Rail {name=VCCpSYZYGY_VIO, 
        voltage=Sensor {name=VCCpSYZYGY_VIO_voltage, value=3.376V}, 
        current=Sensor {name=VCCpSYZYGY_VIO_current, value=0.004A}, 
        power=Sensor {name=VCCpSYZYGY_VIO_power, value=0.0W}}, 
    'VDAC_AVCCAUX': 
        Rail {name=VDAC_AVCCAUX, 
        voltage=Sensor {name=VDAC_AVCCAUX_voltage, value=1.824V}, 
        current=Sensor {name=VDAC_AVCCAUX_current, value=0.001A}, 
        power=Sensor {name=VDAC_AVCCAUX_power, value=0.0W}}, 
    'VADC_AVCC': Rail {name=VADC_AVCC, 
        voltage=Sensor {name=VADC_AVCC_voltage, value=0.952V}, 
        current=Sensor {name=VADC_AVCC_current, value=0.012A}, 
        power=Sensor {name=VADC_AVCC_power, value=0.0W}}, 
    'VCCp1V8': Rail {name=VCCp1V8, 
        voltage=Sensor {name=VCCp1V8_voltage, value=1.816V}, 
        current=Sensor {name=VCCp1V8_current, value=0.61A}, 
        power=Sensor {name=VCCp1V8_power, value=1.0W}}, 
    'VCCp0V85': Rail {name=VCCp0V85, 
        voltage=Sensor {name=VCCp0V85_voltage, value=0.868V}, 
        current=Sensor {name=VCCp0V85_current, value=2.48A}, 
        power=Sensor {name=VCCp0V85_power, value=2.0W}}, 
    'VDAC_AVTT': Rail {name=VDAC_AVTT, 
        voltage=Sensor {name=VDAC_AVTT_voltage, value=2.532V}, 
        current=Sensor {name=VDAC_AVTT_current, value=0.01A}, 
        power=Sensor {name=VDAC_AVTT_power, value=0.04W}}, 
    'VADC_AVCCAUX': Rail {name=VADC_AVCCAUX, 
        voltage=Sensor {name=VADC_AVCCAUX_voltage, value=1.824V}, 
        current=Sensor {name=VADC_AVCCAUX_current, value=0.032A}, 
        power=Sensor {name=VADC_AVCCAUX_power, value=0.06W}}, 
    'VDAC_AVCC': Rail {name=VDAC_AVCC, 
        voltage=Sensor {name=VDAC_AVCC_voltage, value=0.944V}, 
        current=Sensor {name=VDAC_AVCC_current, value=0.004A}, 
        power=Sensor {name=VDAC_AVCC_power, value=0.0W}}, 
    'VCCp3V3': Rail {name=VCCp3V3, 
        voltage=Sensor {name=VCCp3V3_voltage, value=3.328V}, 
        current=Sensor {name=VCCp3V3_current, value=0.36A}, 
        power=Sensor {name=VCCp3V3_power, value=1.0W}}
}
'''